In [1]:
%run nb_dq_utils

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 10, Finished, Available, Finished, True)

In [2]:
from pyspark.sql import functions as F
 
logger = get_logger("silver.product.dq")
# ── Read Bronze sources ───────────────────────────────────────────────
df_product = spark.read.table("lh_adventureworks_bronze.dbo.Production_Product")
df_subcat = spark.read.table("lh_adventureworks_bronze.dbo.Production_ProductSubcategory")
df_cat = spark.read.table("lh_adventureworks_bronze.dbo.Production_ProductCategory")

expected_rows = df_product.count()  
logger.info(f"Product rows      : {expected_rows}")
logger.info(f"Subcategory rows  : {df_subcat.count()}")
logger.info(f"Category rows     : {df_cat.count()}")

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 11, Finished, Available, Finished, False)

INFO:silver.product.dq:Product rows      : 504
INFO:silver.product.dq:Subcategory rows  : 37
INFO:silver.product.dq:Category rows     : 4


In [3]:
# ── Join Product + Subcategory + Category ─────────────────────────────
df_subcat_clean = df_subcat.select(
    F.col("ProductSubcategoryID").alias("sc_ProductSubcategoryID"),
    F.col("ProductCategoryID").alias("sc_ProductCategoryID"),
    F.col("Name").alias("SubcategoryName"),
    F.col("rowguid").alias("sc_rowguid"),
    F.col("ModifiedDate").alias("sc_ModifiedDate")
)

df_cat_clean = df_cat.select(
    F.col("ProductCategoryID").alias("c_ProductCategoryID"),
    F.col("Name").alias("CategoryName"),
    F.col("rowguid").alias("c_rowguid"),
    F.col("ModifiedDate").alias("c_ModifiedDate")
)

# Join product → subcategory
df_silver_product = df_product.join(
    df_subcat_clean,
    df_product["ProductSubcategoryID"] == df_subcat_clean["sc_ProductSubcategoryID"],
    how="left"
).drop("sc_ProductSubcategoryID")

# Join → category
df_silver_product = df_silver_product.join(
    df_cat_clean,
    df_silver_product["sc_ProductCategoryID"] == df_cat_clean["c_ProductCategoryID"],
    how="left"
).drop("c_ProductCategoryID")

# ── Clean up and select final columns ─────────────────────────────────
df_silver_product = df_silver_product.select(
    "ProductID",
    "Name",
    F.col("ProductNumber"),
    F.col("Color"),
    F.col("Size"),
    F.col("Weight"),
    F.col("StandardCost"),
    F.col("ListPrice"),
    F.col("SellStartDate"),
    F.col("SellEndDate"),
    F.col("DiscontinuedDate"),
    F.col("ProductSubcategoryID"),
    F.col("SubcategoryName"),
    F.col("sc_ProductCategoryID").alias("ProductCategoryID"),
    F.col("CategoryName"),
    F.col("ProductLine"),
    F.col("Class"),
    F.col("Style"),
    F.col("SafetyStockLevel"),
    F.col("ReorderPoint"),
    F.col("DaysToManufacture"),
    F.col("rowguid"),
    F.col("ModifiedDate")
)

# ── Type standardization ──────────────────────────────────────────────
df_silver_product = df_silver_product \
    .withColumn("StandardCost", F.col("StandardCost").cast("decimal(19,4)")) \
    .withColumn("ListPrice", F.col("ListPrice").cast("decimal(19,4)")) \
    .withColumn("Weight", F.col("Weight").cast("decimal(8,2)"))
# ── Cache before any checks run ─────────────────────────────────────
df_silver_product.cache()

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 12, Finished, Available, Finished, False)

DataFrame[ProductID: int, Name: string, ProductNumber: string, Color: string, Size: string, Weight: decimal(8,2), StandardCost: decimal(19,4), ListPrice: decimal(19,4), SellStartDate: timestamp, SellEndDate: timestamp, DiscontinuedDate: timestamp, ProductSubcategoryID: int, SubcategoryName: string, ProductCategoryID: int, CategoryName: string, ProductLine: string, Class: string, Style: string, SafetyStockLevel: smallint, ReorderPoint: smallint, DaysToManufacture: int, rowguid: string, ModifiedDate: timestamp]

In [4]:
actual_rows = df_silver_product.count()
 
checks = [
    {
        "name": "Row count",
        "passed": actual_rows == expected_rows,
        "message": f"Expected {expected_rows:,}, got {actual_rows:,}"
    },
    check_null_pk(df_silver_product, "ProductID"),
    check_duplicate_pk(df_silver_product, "ProductID"),
    check_join_coverage(df_silver_product, "ProductSubcategoryID", "SubcategoryName",
                         label="Subcategory join coverage"),
]
 
run_dq_checks(checks, logger, "silver.product")
 

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 13, Finished, Available, Finished, False)

INFO:silver.product.dq:[DQ] [PASS] Row count — Expected 504, got 504
INFO:silver.product.dq:[DQ] [PASS] Null PK — ProductID — 0 nulls
INFO:silver.product.dq:[DQ] [PASS] Duplicate PK — ProductID — 0 duplicates
INFO:silver.product.dq:[DQ] [PASS] Subcategory join coverage — 0 rows with ProductSubcategoryID but no SubcategoryName
INFO:silver.product.dq:[DQ] All checks passed for silver.product.


In [5]:
# ── Create schema and write ───────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.production")
 
df_silver_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.production.product")
 

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 14, Finished, Available, Finished, False)

In [6]:
# ── Verify ────────────────────────────────────────────────────────────
df_verify = spark.read.table("lh_adventureworks_silver.production.product")
logger.info(f"silver.production.product written — {df_verify.count():,} rows verified.")

df_silver_product.unpersist()

StatementMeta(, 03637877-87ab-4620-a4c9-ee98dad3b029, 15, Finished, Available, Finished, False)

INFO:silver.product.dq:silver.production.product written — 504 rows verified.


DataFrame[ProductID: int, Name: string, ProductNumber: string, Color: string, Size: string, Weight: decimal(8,2), StandardCost: decimal(19,4), ListPrice: decimal(19,4), SellStartDate: timestamp, SellEndDate: timestamp, DiscontinuedDate: timestamp, ProductSubcategoryID: int, SubcategoryName: string, ProductCategoryID: int, CategoryName: string, ProductLine: string, Class: string, Style: string, SafetyStockLevel: smallint, ReorderPoint: smallint, DaysToManufacture: int, rowguid: string, ModifiedDate: timestamp]